In [1]:
def column_names():
    """Reads column names for dataframe into array"""
    f = open("kddcup.names.txt")
    s = f.read()
    arr = s.split("\n")[1:-1]
    cols = [a[0:a.index(":")] for a in arr]
    cols.append("target")
    return cols

cols = column_names()

In [2]:
import pandas as pd

kddcup = pd.read_csv('kddcup.data_10_percent_corrected.csv', index_col=False, names=cols)
kddcup.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,target
0,0,tcp,http,SF,181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,normal.
1,0,tcp,http,SF,239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,normal.
2,0,tcp,http,SF,235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
3,0,tcp,http,SF,219,1337,0,0,0,0,...,39,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
4,0,tcp,http,SF,217,2032,0,0,0,0,...,49,1.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,normal.


In [3]:
kddcup.isna().sum()

duration                       0
protocol_type                  0
service                        0
flag                           0
src_bytes                      0
dst_bytes                      0
land                           0
wrong_fragment                 0
urgent                         0
hot                            0
num_failed_logins              0
logged_in                      0
num_compromised                0
root_shell                     0
su_attempted                   0
num_root                       0
num_file_creations             0
num_shells                     0
num_access_files               0
num_outbound_cmds              0
is_host_login                  0
is_guest_login                 0
count                          0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_h

In [4]:
import numpy as np
    
kddcup['target'] = np.where(kddcup['target'] != "normal.", "virus", "normal")
kddcup['target'].value_counts()

target
virus     396743
normal     97278
Name: count, dtype: int64

In [5]:
from sklearn.preprocessing import LabelEncoder

def data_encoder(df, colName):
    encoder = LabelEncoder()
    df[colName] = encoder.fit_transform(df[colName])
    return df

kddcup = data_encoder(kddcup, 'protocol_type')
kddcup = data_encoder(kddcup, 'service')
kddcup = data_encoder(kddcup, 'flag')
kddcup = data_encoder(kddcup, 'target')

kddcup.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,target
0,0,1,22,9,181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,0
1,0,1,22,9,239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,0
2,0,1,22,9,235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,0
3,0,1,22,9,219,1337,0,0,0,0,...,39,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,0
4,0,1,22,9,217,2032,0,0,0,0,...,49,1.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,0


In [6]:
correlations = kddcup.corr()
correlations['target'] = correlations['target'].abs()
correlations.sort_values('target', ascending=False, inplace=True)
correlations['target'][1:]

logged_in                      0.795282
count                          0.752978
dst_host_count                 0.642110
protocol_type                  0.616601
srv_count                      0.566829
dst_host_same_src_port_rate    0.481458
srv_diff_host_rate             0.364687
same_srv_rate                  0.247405
dst_host_srv_serror_rate       0.227975
serror_rate                    0.227739
dst_host_serror_rate           0.227205
srv_serror_rate                0.227189
dst_host_srv_diff_host_rate    0.204958
flag                           0.155672
service                        0.131723
duration                       0.118014
dst_host_diff_srv_rate         0.115901
dst_host_same_srv_rate         0.109950
dst_host_srv_count             0.062566
num_access_files               0.054268
dst_bytes                      0.037709
is_guest_login                 0.032299
wrong_fragment                 0.023630
num_file_creations             0.018671
diff_srv_rate                  0.016522


In [7]:
X = kddcup.drop(['target'], axis=1)
Y = kddcup['target']

In [8]:
from sklearn.preprocessing import StandardScaler

def data_scaling(X):
    """Scales each row in X dataframe to mean and regularized standard deviation"""
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    cols.remove('target')
    X = pd.DataFrame(X, columns=cols)
    return X

X = data_scaling(X)
X.describe()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
count,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,...,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05,4.940210e+05
mean,2.171810e-17,-1.647699e-16,-3.256276e-17,-3.589958e-17,6.508237e-19,4.717573e-18,-7.493462e-18,3.106694e-18,2.277883e-18,-2.035173e-17,...,6.811715e-17,-1.348536e-16,3.352929e-16,5.488494e-17,1.988284e-16,2.807531e-17,3.129707e-17,-5.983263e-17,1.822594e-16,-5.845188e-17
std,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,...,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,-6.779172e-02,-8.115496e-01,-1.729084e+00,-3.484214e+00,-3.061686e-03,-2.628733e-02,-6.673418e-03,-4.772019e-02,-2.571468e-03,-4.413591e-02,...,-3.590542e+00,-1.779188e+00,-1.834994e+00,-2.828667e-01,-1.250621e+00,-1.586293e-01,-4.644176e-01,-4.632024e-01,-2.520395e-01,-2.494640e-01
25%,-6.779172e-02,-8.115496e-01,-6.949824e-01,5.142739e-01,-3.016149e-03,-2.628733e-02,-6.673418e-03,-4.772019e-02,-2.571468e-03,-4.413591e-02,...,3.479668e-01,-1.345391e+00,-8.368938e-01,-2.828667e-01,-1.250621e+00,-1.586293e-01,-4.644176e-01,-4.632024e-01,-2.520395e-01,-2.494640e-01
50%,-6.779172e-02,-8.115496e-01,-6.949824e-01,5.142739e-01,-2.535486e-03,-2.628733e-02,-6.673418e-03,-4.772019e-02,-2.571468e-03,-4.413591e-02,...,3.479668e-01,6.255576e-01,5.993962e-01,-2.828667e-01,8.270476e-01,-1.586293e-01,-4.644176e-01,-4.632024e-01,-2.520395e-01,-2.494640e-01
75%,-6.779172e-02,9.257531e-01,1.373221e+00,5.142739e-01,-2.017381e-03,-2.628733e-02,-6.673418e-03,-4.772019e-02,-2.571468e-03,-4.413591e-02,...,3.479668e-01,6.255576e-01,5.993962e-01,8.323588e-02,8.270476e-01,-1.586293e-01,-4.644176e-01,-4.632024e-01,-2.520395e-01,-2.494640e-01
max,8.234740e+01,2.663056e+00,3.072103e+00,9.585503e-01,7.016400e+02,1.560110e+02,1.498483e+02,2.220663e+01,5.444371e+02,3.831404e+01,...,3.479668e-01,6.255576e-01,5.993962e-01,8.869697e+00,8.270476e-01,2.357583e+01,2.163063e+00,2.162027e+00,4.084676e+00,4.095715e+00


In [9]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectFromModel

def feature_selection(X, Y):
    """Selects highest correlated features from X dataframe"""
    clf = ExtraTreesClassifier(random_state=0).fit(X, Y)
    model = SelectFromModel(clf, prefit=True)
    X_new = model.transform(X)
    return X_new

X_new = feature_selection(X, Y)
X_new.shape

/usr/local/lib/python3.8/dist-packages/sklearn/base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


(494021, 10)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X_new, Y, test_size=0.3, random_state=1)

In [11]:
# from bayes_opt import BayesianOptimization
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import cross_val_score

# def RFC_cv(max_features, n_estimators, X, Y):
#     """Creates RandomForestClassifier model with inputted parameters and runs cross validation
#     to measure accuracy of model with given parameters"""
#     estimator = RandomForestClassifier(
#         max_features = max_features,                       
#         n_estimators = n_estimators
#     )
#     cval = cross_val_score(estimator, X, Y, cv=4)
#     return cval.mean()

# def optimize_RFC(X, Y):
#     """Uses Bayesian optimization function to narrow down most accurate values for parameters
#     and returns parameters from model with highest accuracy"""
#     def RFC_crossval(max_features, n_estimators):
#         """Wrapper that converts parameters into correct format and returns model results
#         from RFC_cv()"""
#         return rfc_cv(
#             max_features = 'log2' if max_features < 1 else 'sqrt',                     
#             n_estimators = int(n_estimators),
#             X = X,
#             Y = Y
#         )
    
#     optimizer = BayesianOptimization(
#        f=rfc_crossval,
#         pbounds={
#             # "max_depth": (10, 100),
#             "max_features": (0, 2),
#             # "min_samples_leaf": (1, 4),
#             # "min_samples_split": (2, 10),
#             "n_estimators": (10, 250),
#         },
#         random_state=1
#     )
#     optimizer.maximize(init_points = 3, n_iter=5)
#     print(optimizer.max)
#     return optimizer.max['params']

In [12]:
import time

def train(name, model_type, X, Y):
    """Fits model off of training data and returns them"""
    print(f"Model: {name}")
    name = model_type(random_state=1) if name != "Naive Bayes" and name != "K Nearest Neighbors" else model_type()
    
    start = time.time()
    name.fit(X, Y)
    end = time.time()
    print(f"Time to train: {end-start}")
    return name, end-start

In [13]:
from sklearn import metrics
from sklearn.model_selection import cross_val_score

def test(name, model, X, Y):
    """Runs model on testing data and returns variety of tests to show accuracy
    of model on testing data, including cross validation"""
    start = time.time()
    predictions = model.predict(X)
    end = time.time()
    print(f"Time to test: {end-start}")
    
    # accuracy = metrics.accuracy_score(Y, predictions)
    # confusion_matrix = metrics.confusion_matrix(Y, predictions)
    # classification = metrics.classification_report(Y, predictions)
    f1 = metrics.f1_score(Y, predictions)
    scores = cross_val_score(model, X, Y, cv=5)
    # print(f"Accuracy score: {accuracy}")
    # print(f"Confusion matrix: \n{confusion_matrix}")
    # print(f"Classification: \n{classification}")
    print(f"F1 score: {f1}")
    print(f"Cross validation mean score: {scores.mean()}\n")

    return pd.DataFrame([[name, end-start, f1, scores.mean()]], columns=['name', 'test time', 'F1 score', 'mean cv accuracy'])

In [16]:
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.layers import Dense

def neural_network(X_train, Y_train, X_test, Y_test):
    ANN = Sequential()
    ANN.add(Dense(128, activation='relu', input_dim=X_train.shape[1]))
    ANN.add(Dense(1, activation='sigmoid'))
    start = time.time()
    ANN.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='val_loss', patience=3)
    history = ANN.fit(X_train, Y_train, epochs=50, validation_split=0.2, callbacks=[early_stopping])
    end = time.time()
    return history, pd.DataFrame([["Neural network", end-start, history.history[]]], columns=['name', 'mean cv accuracy'])

In [17]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC  

models = []
models.append(('Gradient Boosting', GradientBoostingClassifier))
models.append(('Random Forest', RandomForestClassifier))
models.append(('Logistic Regression', LogisticRegression))
models.append(('Naive Bayes', GaussianNB))
models.append(('K Nearest Neighbors', KNeighborsClassifier))
models.append(('Decision Trees', DecisionTreeClassifier))
models.append(('Support Vector Machine', SVC))

test_df = pd.DataFrame(columns=['name', 'test time', 'f1 score', 'mean cv accuracy'])

history, test_data = neural_network(X_train, Y_train, X_test, Y_test)

for name, model_type in models:
    model = train(name, model_type, X_train, Y_train)
    test_data = test(name, model, X_test, Y_test)
    test_df = pd.concat([test_df, test_data])



Epoch 1/50
8646/8646 [==============================] - 14s 2ms/step - loss: 0.0336 - accuracy: 0.9896 - val_loss: 0.0277 - val_accuracy: 0.9916
Epoch 2/50
8646/8646 [==============================] - 13s 2ms/step - loss: 0.0250 - accuracy: 0.9920 - val_loss: 0.0242 - val_accuracy: 0.9925
Epoch 3/50
8646/8646 [==============================] - 13s 2ms/step - loss: 0.0231 - accuracy: 0.9924 - val_loss: 0.0230 - val_accuracy: 0.9924
Epoch 4/50
8646/8646 [==============================] - 13s 2ms/step - loss: 0.0223 - accuracy: 0.9926 - val_loss: 0.0267 - val_accuracy: 0.9922
Epoch 5/50
8646/8646 [==============================] - 13s 2ms/step - loss: 0.0216 - accuracy: 0.9926 - val_loss: 0.0212 - val_accuracy: 0.9931
Epoch 6/50
8646/8646 [==============================] - 13s 2ms/step - loss: 0.0210 - accuracy: 0.9928 - val_loss: 0.0235 - val_accuracy: 0.9929
Epoch 7/50
8646/8646 [==============================] - 13s 2ms/step - loss: 0.0207 - accuracy: 0.9928 - val_loss: 0.0195 - val_ac

TypeError: If no scoring is specified, the estimator passed should have a 'score' method. The estimator <keras.src.engine.sequential.Sequential object at 0x7f1dcdd727c0> does not.

In [ ]:
import plotly.express as px

px.scatter(test_df, x='F1 score', y='mean cv accuracy', color='test time', hover_data='name').show()

0.9923485403145189
